# d66 — TS3/TS4 role-mapping regression and fix

The `cat_SO2Me_atom_5` example exposes the former `tsguess2` error and confirms the FRUST fix.

**Cause:** the generic `[#5]~[#1]~[#5]~[#6]` pattern did not distinguish catalyst-B from HBpin-B and could select the catalyst aryl carbon instead of the substrate carbon.

In [1]:
import math

import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Geometry import Point3D

import frust as ft
from frust.tsguess2.api import PRUNE_RMS_THRESH, RANDOM_SEED

system = pd.DataFrame([{
    "system_name": "dimethoxybenzene_1_3__cat_SO2Me_atom_5",
    "substrate_name": "dimethoxybenzene_1_3",
    "catalyst_name": "cat_SO2Me_atom_5",
    "substrate_smiles": "COC1=CC=CC(OC)=C1",
    "catalyst_smiles": "CN(C)c1ccccc1BS(C)(=O)=O",
    "rpos": 3,
}])

generated = ft.tsguess2.create_ts_guess_dataframes(
    system,
    ts_types=["TS3", "TS4"],
    n_confs=1,
    spec_profile="r2scan3c",
    spec_match="exact",
)
fixed = pd.concat(generated.values(), ignore_index=True)
generic_query = Chem.MolFromSmarts("[#5]~[#1]~[#5]~[#6]")


def reembed(row, roles):
    """Embed a row using a supplied role mapping."""
    mol = Chem.AddHs(Chem.MolFromSmiles(row["smiles"]))
    spec = ft.tsguess2.resolve_profile_spec(
        row["state_id"], "r2scan3c", match="exact"
    ).spec
    coord_map = {
        roles[role]: Point3D(*xyz)
        for role, xyz in spec.role_coordinates.items()
    }
    cid = list(AllChem.EmbedMultipleConfs(
        mol,
        numConfs=1,
        maxAttempts=0,
        randomSeed=RANDOM_SEED,
        useRandomCoords=False,
        pruneRmsThresh=PRUNE_RMS_THRESH,
        coordMap=coord_map,
        ignoreSmoothingFailures=True,
        enforceChirality=True,
        useSmallRingTorsions=True,
        numThreads=1,
    ))[0]
    conf = mol.GetConformer(cid)
    result = row.copy()
    result["constraint_roles"] = roles
    result["coords_embedded"] = [
        tuple(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())
    ]
    return result


legacy_rows = []
audit = []
role_order = ("cat_B", "transfer_H", "pin_B", "substrate_C")
for _, row in fixed.iterrows():
    mol = Chem.AddHs(Chem.MolFromSmiles(row["smiles"]))
    matches = mol.GetSubstructMatches(generic_query)
    legacy_roles = dict(zip(role_order, matches[0]))
    legacy_rows.append(reembed(row, legacy_roles))
    audit.append({
        "state": row["state_id"],
        "generic matches": matches,
        "former selection": tuple(legacy_roles[role] for role in role_order),
        "fixed FRUST selection": tuple(row["constraint_roles"][role] for role in role_order),
    })

legacy = pd.DataFrame(legacy_rows)
pd.DataFrame(audit)

,state,generic matches,former selection,fixed FRUST selection
0,TS3,"((10, 11, 12, 13), (10, 11, 12, 7))","(10, 11, 12, 13)","(12, 11, 10, 7)"
1,TS4,"((10, 11, 12, 13), (10, 11, 12, 7))","(10, 11, 12, 13)","(12, 11, 10, 7)"


## 1. Former behavior

The cyan labels show the error: `cat_B` is on HBpin, `pin_B` is on the catalyst, and `substrate_C` is on the catalyst aryl ring.

In [2]:
ft.vis.show_scene(ft.vis.ts_guess_scene_from_dataframe(
    legacy,
    show_roles=True,
    show_constraint_distances=True,
    columns=2,
    cell_size=(475, 420),
))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 2. Fix now used by FRUST

HBpin-B is identified by its two oxygen neighbours. The other boron is catalyst-B; their shared H and C are the transferring H and substrate C. FRUST encodes these environments in a directed SMARTS and validates the result.

In [3]:
def assign_ts34_roles_by_topology(mol):
    """Return chemically directed roles for a connected TS3/TS4 molecule."""
    borons = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() == 5]
    pin_b = [
        idx for idx in borons
        if sum(n.GetAtomicNum() == 8 for n in mol.GetAtomWithIdx(idx).GetNeighbors()) == 2
    ]
    if len(borons) != 2 or len(pin_b) != 1:
        raise ValueError("Expected two borons and one B bonded to two oxygens")
    pin_b = pin_b[0]
    cat_b = next(idx for idx in borons if idx != pin_b)
    shared = (
        {n.GetIdx() for n in mol.GetAtomWithIdx(cat_b).GetNeighbors()}
        & {n.GetIdx() for n in mol.GetAtomWithIdx(pin_b).GetNeighbors()}
    )
    transfer_h = [idx for idx in shared if mol.GetAtomWithIdx(idx).GetAtomicNum() == 1]
    substrate_c = [idx for idx in shared if mol.GetAtomWithIdx(idx).GetAtomicNum() == 6]
    if len(transfer_h) != 1 or len(substrate_c) != 1:
        raise ValueError("Expected one shared H and one shared C")
    return {
        "cat_B": cat_b,
        "transfer_H": transfer_h[0],
        "pin_B": pin_b,
        "substrate_C": substrate_c[0],
    }


def measured_over_target(row, roles, left, right):
    """Format a measured role distance and its template target."""
    target = next(
        entry["value"]
        for entry in row["constraint_spec"]
        if entry["kind"] == "distance" and set(entry["roles"]) == {left, right}
    )
    measured = math.dist(
        row["coords_embedded"][roles[left]],
        row["coords_embedded"][roles[right]],
    )
    return f"{measured:.2f} / {target:.2f}"


records = []
for version, rows in (("former", legacy), ("fixed", fixed)):
    for _, row in rows.iterrows():
        mol = Chem.AddHs(Chem.MolFromSmiles(row["smiles"]))
        chemical_roles = assign_ts34_roles_by_topology(mol)
        if version == "fixed":
            assert row["constraint_roles"] == chemical_roles
        records.append({
            "state": row["state_id"],
            "geometry": version,
            "stored tuple": tuple(row["constraint_roles"][role] for role in role_order),
            "catB-Csub": measured_over_target(row, chemical_roles, "cat_B", "substrate_C"),
            "pinB-Csub": measured_over_target(row, chemical_roles, "pin_B", "substrate_C"),
            "catB-H": measured_over_target(row, chemical_roles, "cat_B", "transfer_H"),
            "pinB-H": measured_over_target(row, chemical_roles, "pin_B", "transfer_H"),
        })

pd.DataFrame(records)

,state,geometry,stored tuple,catB-Csub,pinB-Csub,catB-H,pinB-H
0,TS3,former,"(10, 11, 12, 13)",1.46 / 1.56,1.67 / 2.63,1.28 / 1.57,1.58 / 1.22
1,TS4,former,"(10, 11, 12, 13)",1.51 / 1.80,1.58 / 1.60,1.64 / 1.23,1.23 / 1.67
2,TS3,fixed,"(12, 11, 10, 7)",1.52 / 1.56,2.23 / 2.63,1.53 / 1.57,1.21 / 1.22
3,TS4,fixed,"(12, 11, 10, 7)",1.65 / 1.80,1.55 / 1.60,1.22 / 1.23,1.67 / 1.67


Distances are **measured / r2SCAN-3c template** in Å. The fixed mapping is `(12, 11, 10, 7)` and the B–H distances now follow the intended TS3/TS4 core.

In [4]:
ft.vis.show_scene(ft.vis.ts_guess_scene_from_dataframe(
    fixed,
    show_roles=True,
    show_constraint_distances=True,
    columns=2,
    cell_size=(475, 420),
))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Result

The regression is fixed before optimization. FRUST now uses chemically directed SMARTS, validates HBpin connectivity, and raises rather than silently accepting ambiguous TS3/TS4/INT3 mappings. Existing affected calculations must still be regenerated.